In [ ]:

MODEL_NAME ="meta-llama/Llama-2-7b-chat-hf"#"openai-community/gpt2" 

import os
import gc
import re
import json
import random
import string
import torch
import numpy as np
#import evaluate
import random
from tqdm import tqdm
from functools import partial
from tabulate import tabulate
import plotly.express as px
import huggingface_hub
import torch.nn.functional as F
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from huggingface_hub import login
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge
from datasets import load_dataset

login(token="enter_token_from_bucket")

torch.set_grad_enabled(True)                                             
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = TransformerBridge.boot_transformers(
    MODEL_NAME,
    dtype=torch.float16,
    n_devices=2,)

In [ ]:


#Checks whether the generated text contains any acceptable answer.
def is_correct(text: str, options: list[str]) -> bool:
    
    def _lowercase_only(s: str) -> str:
        return s.lower()
    
    
    def _normalize(s: str) -> str:
        s = s.lower()
        punctuation_to_strip = string.punctuation.replace(".", "").replace("-", "")
        s = s.translate(str.maketrans("", "", punctuation_to_strip))
        s = re.sub(r'(?<!\d)\.(?!\d)', '', s)   # drop '.' unless part of a number
        s = re.sub(r'(?<!\d)-(?!\d)', ' ', s)   # drop '-' unless part of a number
        s = re.sub(r"\s+", " ", s).strip()
        return s
    
    
    def contains_phrase(haystack: str, phrase: str) -> bool:
        if not phrase:
            return False
        pattern = r"\b" + re.escape(phrase) + r"\b"
        return bool(re.search(pattern, haystack))

    if not text or not options:
        return False

    # --- Stage 1: lowercase only ---
    text_lower = _lowercase_only(text)
    if any(contains_phrase(text_lower, _lowercase_only(opt)) for opt in options):
        return True

    # --- Stage 2: full normalization (fallback) ---
    text_norm = _normalize(text)
    if any(contains_phrase(text_norm, _normalize(opt)) for opt in options):
        return True

    return False


#Builds a prompt from the (substituted) context and the question.
def build_conflict_prompt(row):

    context = row["sub_context"]
    question = row["question"]
    prompt = f"{context}\n\nQuestion: {question}\nAnswer in few words:"

    return prompt


In [ ]:
#Generates an answer for each example and sorts it into the memory-answered or context-answered list, saving both to JSON.
def answer_from_mem_or_context(dataset_arg):
  list_of_memory_answers_data = []
  list_of_context_answers_data = []

  data_QA=dataset_arg
  count=0
  for dicti in data_QA:
      print("in i=",count)
      count+=1
      prompt = build_conflict_prompt(dicti)
      input_ids = model . to_tokens ( prompt , prepend_bos = True )


      full_tokens = model.generate(
          input_ids,
          max_new_tokens=15,
          do_sample=False  
      )

      output=model.to_string(full_tokens)[0]
      #print("output",output)
      bos_str = model.to_string([model.tokenizer.bos_token_id])
      bos_len = len(bos_str)
      generated_only = output[len(prompt)+ bos_len:]
      print(generated_only)
      if is_correct(generated_only,dicti["org_answer"]):
          list_of_memory_answers_data.append(dicti)
      if is_correct(generated_only,dicti["sub_answer"]):
          list_of_context_answers_data.append(dicti)

      with open(file_path_memory_answers, "w") as f:
          json.dump(list_of_memory_answers_data, f)

      with open(file_path_context_answers, "w") as f:
          json.dump(list_of_context_answers_data, f)

      gc.collect()
      torch.cuda.empty_cache()

      return list_of_memory_answers_data,list_of_context_answers_data

In [ ]:
data = np.load('/kaggle/input/datasets/yochaqwami/mem-contetxt-14-9-with-prob-sum/data_proccesed_mem_metrics_2.npz')
total_MEM_MEM_A = torch.from_numpy(data['total1'])
total_context_MEM_A= torch.from_numpy(data['total2'])
total_diff_MEM_A= torch.from_numpy(data['total_diff'])
total_multi_MEM_A= torch.from_numpy(data['total_multi'])

data = np.load('/kaggle/input/datasets/yochaqwami/mem-contetxt-14-9-with-prob-sum/data_proccesed_context_metrics_2.npz')
total_context_context_A = torch.from_numpy(data['total1'])
total_MEM_context_A= torch.from_numpy(data['total2'])
total_diff_context_A= torch.from_numpy(data['total_diff'])
total_multi_context_A= torch.from_numpy(data['total_multi'])

In [ ]:
#extractig  memory answers and context answers from Knownfact NQ-swap dataset we created
fact_NQ_swap=[]
with open('/kaggle/input/datasets/nachmanrog/known-facts-nq-swap/nqswap_style_dataset.json', "r") as f:
    fact_NQ_swap = json.load(f)

list_of_memory_answers_data,list_of_context_answers_data=answer_from_mem_or_context(ds_NQ_Swap)

In [ ]:
file_path_memory_answers = "/kaggle/input/datasets/nachmanrog/fact-nq-mem-context/list_of_memory_answers_data_fact_nq.json"
file_path_context_answers = "/kaggle/input/datasets/nachmanrog/fact-nq-mem-context/list_of_context_answers_data_fact_nq.json"

list_of_memory_answers_data=[]
list_of_context_answers_data=[]
if os.path.exists(file_path_memory_answers):
    with open(file_path_memory_answers, "r") as f:
        list_of_memory_answers_data = json.load(f)


if os.path.exists(file_path_context_answers):
    with open(file_path_context_answers, "r") as f:
        list_of_context_answers_data = json.load(f)

In [ ]:
#Formats a SQuAD example as context, question and "Answer:".
def format_squad_prompt(example):
    return f"Context: {example['context']}\nQuestion: {example['question']}\nAnswer:"
#Formats a SQuAD example with the question only, no context.
def format_squad_prompt_only_Q(example):
    return f"Question: {example['question']}\nAnswer:"

In [ ]:
#Returns a reproducible random pick from the model's top-k next-token predictions.
def get_random_top_k_token_id(model, text, k=5, seed=42, prepend_bos=True,
                              weighted=False, return_string=False):
    tokens = model.to_tokens(text, prepend_bos=prepend_bos)
    model.reset_hooks(including_permanent=True)
    with torch.no_grad():
        logits = model(tokens)                                

    probs = torch.softmax(logits[0, -1].float(), dim=-1)
    top_probs, top_ids = probs.topk(k)                        


    gen = torch.Generator().manual_seed(seed)
    if weighted:
        idx = torch.multinomial(top_probs.cpu(), 1, generator=gen).item()
    else:
        idx = torch.randint(0, k, (1,), generator=gen).item()

    chosen_id = top_ids[idx].item()

    if return_string:
        return chosen_id, model.to_string(chosen_id)
    return chosen_id

In [ ]:
#Loads SQuAD and keeps only short-context examples for training and validation.
def prepare_squad_datasets(seed= 42,max_context_words = 100,n_train= 1500,n_val= 10):

    from datasets import load_dataset
 
    squad_train = load_dataset("rajpurkar/squad", split="train").shuffle(seed=seed)
    squad_val = load_dataset("rajpurkar/squad", split="validation").shuffle(seed=seed)
 
    filtered_train = squad_train.filter(lambda ex: len(ex["context"].split()) < max_context_words)
    filtered_val = squad_val.filter(lambda ex: len(ex["context"].split()) < max_context_words)
 
    train_hf = filtered_train.select(range(n_train))
    val_hf = filtered_val.select(range(n_val))
    return train_hf, val_hf

In [ ]:
#Loss functions

#Pushes the suppressed heads' outputs toward zero (MSE).
def multi_head_target_loss(cache_head_last_pos, model):
    d_model = model.cfg.d_model
    losses = []
    for (layer, head) in cache_head_last_pos.keys():
        vec = cache_head_last_pos[(layer, head)]
        target_embed = torch.zeros(d_model, device=vec.device, dtype=vec.dtype)
        loss = F.mse_loss(vec, target_embed.expand_as(vec))
        losses.append(loss.to("cuda:1")) 

    total = sum(losses) / len(losses)
    return total.to(model.cfg.dtype)
    
#Pushes the suppressed heads' logit-lens distribution toward the BOS token (attention-sink style).
def multi_head_target_loss_bos(cache_head_last_pos, model):
    bos_id = model.tokenizer.bos_token_id
    losses = []
    for (layer, head) in cache_head_last_pos.keys():
        vec = cache_head_last_pos[(layer, head)]  

        logits = model.unembed(model.ln_final(vec))  
        log_probs = F.log_softmax(logits, dim=-1)
        target_log_prob = log_probs[:, bos_id]  

        loss = -target_log_prob.mean()
        losses.append(loss.to("cuda:1"))  

    total = sum(losses) / len(losses)
    return total.to(model.cfg.dtype)
    
#Makes each amplified head more confident in its own top predicted token.   
def multi_head_self_amplification_loss_soft(cache_head_last_pos, model):
    losses = []
    for (layer, head) in cache_head_last_pos.keys():
        h = cache_head_last_pos[(layer, head)]
        normed = model.ln_final(h)
        logits = model.unembed(normed)
        log_probs = F.log_softmax(logits, dim=-1)

        top_log_prob = log_probs.max(dim=-1).values  
        losses.append(-top_log_prob.mean())
    return sum(losses) / len(losses)

#Cross-entropy on the final-layer prediction against the target answer token.
def logit_log_prob_loss(logits, target_token_ids):
    # logits: [B, T, vocab]
    last_logits = logits[:, -1, :]                     
    log_probs = F.log_softmax(last_logits, dim=-1)
    batch_size = logits.shape[0]
    target_log_prob = log_probs[torch.arange(batch_size), target_token_ids]
    return -target_log_prob.mean().to("cuda:1").to(model.cfg.dtype)

#Pulls each soft-prompt vector toward its nearest vocabulary embedding by cosine similarity, and returns the nearest token indices.
def nearest_embedding_loss(soft_prompt, W_E):
    # compute in fp32 for precision, cast down at the end
    soft_norm = F.normalize(soft_prompt.float(), p=2, dim=-1)  
    WE_norm = F.normalize(W_E.float(), p=2, dim=-1)        

    with torch.no_grad():

        sim = torch.matmul(soft_norm, WE_norm.T)
        nearest_idx = sim.argmax(dim=-1) 

    nearest_embed = W_E[nearest_idx].float()                          
    nearest_embed_norm = F.normalize(nearest_embed, p=2, dim=-1)     


    cos_sim = (soft_norm * nearest_embed_norm).sum(dim=-1)   
    loss = (1.0 - cos_sim).mean()*10
    
    return loss.to("cuda:1").to(model.cfg.dtype), nearest_idx


#Injects the soft-prompt vectors into the embedding output at the virtual-token positions.
def hook_fn(value, hook,soft_prompt=None, n_virtual_tokens=None):

    batch_size = value.shape[0]
    prompt = soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)  

    value = value.clone()  
    value[:, 1:n_virtual_tokens+1, :] = prompt 
    return value


#Registers hooks that save the last-position output of the chosen heads.
def add_hook_on_heads_res(model,heads_to_save,dict_head_res):

    def result_head(res, hook, layer_head,dict_r):
        dict_r[layer_head]=res[:,-1,layer_head[1],:]  
        return res


    for layer, head in heads_to_save:
        model.add_hook(
            utils.get_act_name("result", layer),
            partial(result_head, layer_head=(layer, head),dict_r=dict_head_res),
            is_permanent=True,)

In [ ]:
#Returns the (layer, head) indices of the top-k values in a matrix.
def list_of_top_k_heads(mat,k):
    flat_values, flat_indices = torch.topk(mat.flatten(), k=k)
    row_indices, col_indices = torch.unravel_index(flat_indices, mat.shape)
    top_k_heads_of_mem = torch.stack([row_indices, col_indices], dim=1)
    return top_k_heads_of_mem
#Returns the top-k heads among those with a positive difference score.
def top_k_positive_heads(diff_tensor, k) :

    positive = diff_tensor * (diff_tensor > 0)
    _, flat_indices = torch.topk(positive.flatten(), k=k)
    row_indices, col_indices = torch.unravel_index(flat_indices, diff_tensor.shape)
    return torch.stack([row_indices, col_indices], dim=1)

#Splits two head sets into "a only", "b only" and "common".
def partition_heads(heads_a, heads_b) :

    set_a = set(map(tuple, heads_a.tolist()))
    set_b = set(map(tuple, heads_b.tolist()))
    common = set_a & set_b
    return {"a_only": set_a - common, "b_only": set_b - common, "common": common}

In [ ]:
#Builds input ids as BOS, then placeholder tokens for the soft prompt, then the real text.
def build_virtual_token_input_ids(model, batch_texts, n_virtual_tokens, placeholder_id=model.tokenizer.pad_token_id):

    bos_id = model.tokenizer.bos_token_id
    if placeholder_id is None:
        placeholder_id = model.tokenizer.pad_token_id
 
    real_input_ids = model.to_tokens(batch_texts, prepend_bos=False)
    batch_size = real_input_ids.shape[0]
 
    bos_tensor = torch.full((batch_size, 1), bos_id, device=model.cfg.device, dtype=torch.long)
    placeholders = torch.full((batch_size, n_virtual_tokens), placeholder_id, device=model.cfg.device, dtype=torch.long)
    return torch.cat([bos_tensor, placeholders, real_input_ids], dim=1)

In [ ]:
#Total loss from head suppression (MSE), context-head amplification and embedding proximity.
def compute_combined_loss(model, head_res_dict: dict, head_res_context_dict: dict,logits,target_token, soft_prompt):
    #loss_align=logit_log_prob_loss(logits, target_token)
    loss_task = multi_head_target_loss(head_res_dict, model)
    loss_amp_context = multi_head_self_amplification_loss_soft(head_res_context_dict, model)
    loss_embed, _ = nearest_embedding_loss(soft_prompt, model.embed.W_E.to(soft_prompt.device))
    total = loss_task + loss_embed + loss_amp_context
    return total, {"task": loss_task, "embed": loss_embed, "amp_context": loss_amp_context}
#Same, but with the alignment loss instead of embedding proximity.
def compute_combined_loss_2(model, head_res_dict: dict, head_res_context_dict: dict,logits,target_token, soft_prompt: torch.nn.Parameter):
    loss_align=logit_log_prob_loss(logits, target_token)
    loss_task = multi_head_target_loss(head_res_dict, model)
    loss_amp_context = multi_head_self_amplification_loss_soft(head_res_context_dict, model)

    total = loss_task + loss_amp_context+loss_align
    return total, {"task": loss_task, "loss_align": loss_align, "amp_context": loss_amp_context}
#Same, with all four terms (MSE suppression, amplification, embedding proximity and alignment).
def compute_combined_loss_3(model, head_res_dict: dict, head_res_context_dict: dict,logits,target_token, soft_prompt: torch.nn.Parameter):
    loss_align=logit_log_prob_loss(logits, target_token)
    loss_task = multi_head_target_loss(head_res_dict, model)
    loss_amp_context = multi_head_self_amplification_loss_soft(head_res_context_dict, model)
    loss_embed, _ = nearest_embedding_loss(soft_prompt, model.embed.W_E.to(soft_prompt.device))
    total = loss_task + loss_embed + loss_amp_context+loss_align
    return total, {"task": loss_task, "embed": loss_embed, "loss_align": loss_align, "amp_context": loss_amp_context}
#Same as 3, but suppression uses the BOS-target loss.
def compute_combined_loss_4(model, head_res_dict: dict, head_res_context_dict: dict,logits,target_token, soft_prompt: torch.nn.Parameter):
    loss_align=logit_log_prob_loss(logits, target_token)
    loss_task = multi_head_target_loss_bos(head_res_dict, model)
    loss_amp_context = multi_head_self_amplification_loss_soft(head_res_context_dict, model)
    loss_embed, _ = nearest_embedding_loss(soft_prompt, model.embed.W_E.to(soft_prompt.device))
    total = loss_task + loss_embed + loss_amp_context+loss_align
    return total, {"task": loss_task, "embed": loss_embed, "loss_align": loss_align, "amp_context": loss_amp_context}

#Same as 4, without the embedding-proximity term.
def compute_combined_loss_4_no_embedd(model, head_res_dict: dict, head_res_context_dict: dict,logits,target_token, soft_prompt: torch.nn.Parameter):
    loss_align=logit_log_prob_loss(logits, target_token)
    loss_task = multi_head_target_loss_bos(head_res_dict, model)
    loss_amp_context = multi_head_self_amplification_loss_soft(head_res_context_dict, model)
    total = loss_task  + loss_amp_context+loss_align
    return total, {"task": loss_task, "loss_align": loss_align, "amp_context": loss_amp_context}
#Runs one forward pass with the soft prompt and head hooks attached, and returns the loss.
def forward_and_loss(model,soft_prompt,n_virtual_tokens,target_heads,target_heads_context,prompt_text,target_token,compute_combined_loss=compute_combined_loss):
    head_res_dict = {}
    head_res_context_dict = {}
 
    model.reset_hooks(including_permanent=True)
    add_hook_on_heads_res(model, target_heads, head_res_dict)
    add_hook_on_heads_res(model, target_heads_context, head_res_context_dict)
    model.add_hook(
        "hook_embed",
        partial(hook_fn, soft_prompt=soft_prompt, n_virtual_tokens=n_virtual_tokens),
        is_permanent=True,
    )
 
    input_ids = build_virtual_token_input_ids(model, [prompt_text], n_virtual_tokens)
    logits=model(input_ids) 
    return compute_combined_loss(model, head_res_dict, head_res_context_dict, logits,target_token,soft_prompt)



In [ ]:
#Computes the average validation loss of a soft prompt.
def evaluate_soft_prompt_loss(model,val_data,soft_prompt,n_virtual_tokens,target_heads,target_heads_context,compute_combined_loss=compute_combined_loss):
    total = 0.0
    with torch.no_grad():
        for data in val_data:
            token_id_answer=[model.to_tokens( data['answers']['text'] , prepend_bos = False )[0,0].item()]

            prompt_text = format_squad_prompt(data)
            loss, _ = forward_and_loss(model, soft_prompt, n_virtual_tokens, target_heads, target_heads_context, prompt_text,token_id_answer,compute_combined_loss=compute_combined_loss)
            total += loss.item()
    model.reset_hooks(including_permanent=True)
    gc.collect()
    torch.cuda.empty_cache()
    return total / len(val_data)

In [ ]:
#Trains the soft prompt with periodic validation, checkpointing the best one.
def train_soft_prompt(model,train_data,val_data,target_heads,target_heads_context,n_virtual_tokens,lr = 1e-3,eps = 1e-4,val_every = 100,patience = 100_000,num_of_epochs=1,checkpoint_path = "best_soft.pt",log_every= 10,compute_combined_loss=compute_combined_loss) :
    torch.set_grad_enabled(True)
    model.set_use_attn_result(True)
    model.requires_grad_(False)
 
    hidden_dim = model.cfg.d_model
    soft_prompt = torch.nn.Parameter(
        torch.randn(n_virtual_tokens, hidden_dim, dtype=model.cfg.dtype, device=model.cfg.device) * 0.02
    )
    optimizer = torch.optim.AdamW([soft_prompt], lr=lr, eps=eps)
 
    train_loss_history: List[tuple] = []
    val_loss_history: List[tuple] = []
    min_loss = float("inf")
    patience_countdown = 0

    for ep in range(num_of_epochs):
        for i, data in enumerate(train_data):
            try:
                token_id_answer=[model.to_tokens( data['answers']['text'] , prepend_bos = False )[0,0].item()]
                print(token_id_answer)
                prompt_text = format_squad_prompt(data)
                loss, l_d = forward_and_loss(model, soft_prompt, n_virtual_tokens, target_heads, target_heads_context, prompt_text,token_id_answer,compute_combined_loss=compute_combined_loss)
                print(l_d)
                if i % log_every == 0:
                    train_loss_history.append((i, loss.item()))
         
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
         
                model.reset_hooks(including_permanent=True)
                model.zero_grad(set_to_none=True)
                optimizer.zero_grad(set_to_none=True)
                gc.collect()
                torch.cuda.empty_cache()
         
                patience_countdown += 1
                if loss.item() < min_loss:
                    min_loss = loss.item()
                    patience_countdown = 0
                    torch.save(soft_prompt.detach().cpu(), checkpoint_path)
                if patience_countdown >= patience:
                    break
         
                if i % val_every == 0:
                    avg_val_loss = evaluate_soft_prompt_loss(
                        model, val_data, soft_prompt, n_virtual_tokens, target_heads, target_heads_context,compute_combined_loss=compute_combined_loss)
                    val_loss_history.append((i, avg_val_loss))
                    print("val_loss=", avg_val_loss)
            except Exception as e:
                print("!!!!!!!!!skip!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
                model.zero_grad(set_to_none=True)
                optimizer.zero_grad(set_to_none=True)
                gc.collect()
                torch.cuda.empty_cache()
     
    best_soft_prompt = torch.nn.Parameter(
        torch.load(checkpoint_path).to(model.cfg.device).to(model.cfg.dtype)
    )
    return best_soft_prompt, train_loss_history, val_loss_history

In [ ]:
#Projects each soft-prompt vector to its nearest vocabulary token id (the hard prompt).
def soft_prompt_to_ids(soft_prompt, model, k = 5) :

    W_E = model.embed.W_E
    vec_norm = F.normalize(soft_prompt, dim=-1)
    W_E_norm = F.normalize(W_E, dim=-1)
    cos_sim = vec_norm @ W_E_norm.T
    _, top_indices = torch.topk(cos_sim, k=k, dim=-1)
    return top_indices[:, 0].to(model.cfg.device)
 
#Generates a reproducible random token-id prompt as a control. 
def random_token_ids(vocab_size, length, seed = 42, device = model.cfg.device):
    g = torch.Generator(device=device).manual_seed(seed)
    return torch.randint(low=0, high=vocab_size, size=(length,), generator=g, device=device)

In [ ]:
#Labels an output as "mem", "context" or "neither".
def classify_generation(output_text, org_answer, sub_answer):
    if is_correct(output_text, org_answer):
        return "mem"
    if is_correct(output_text, sub_answer):
        return "context"
    return "neither"

#Prepends a discrete prompt to each test example, generates, and counts mem, context and neither.
def evaluate_conflict_completions(model,prompt_token_ids,eval_data,repeat= 1,max_new_tokens= 50):

    counts = {"mem": 0, "context": 0, "neither": 0}
    bos_id = model.tokenizer.bos_token_id
    prompt_block = prompt_token_ids.unsqueeze(0)
 
    for data in eval_data:
        batch = [build_conflict_prompt(data)]
        real_input_ids = model.to_tokens(batch, prepend_bos=False)
        batch_size = real_input_ids.shape[0]
 
        bos_tensor = torch.full((batch_size, 1), bos_id, device=model.cfg.device, dtype=torch.long)
        repeated_block = prompt_block.expand(batch_size, -1).repeat(1, repeat)
        input_ids = torch.cat([bos_tensor, repeated_block, real_input_ids], dim=1).to(model.cfg.device)
 
        pre_output = model.to_string(input_ids)[0]
        with torch.no_grad():
            full_tokens = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
        output = model.to_string(full_tokens)[0][len(pre_output):]
 
        counts[classify_generation(output, data["org_answer"], data["sub_answer"])] += 1
        print("output->",output)
        print(counts)
        gc.collect()
        torch.cuda.empty_cache()
 
    return counts
 
#Same, but feeds the raw continuous soft prompt to a HuggingFace model, with no discretization. 
def evaluate_conflict_completions_with_embeddings_hf(model,tokenizer,soft_prompt,eval_data,repeat = 1,max_new_tokens: int = 50):

    counts = {"mem": 0, "context": 0, "neither": 0}
    device = soft_prompt.device
    embed_layer = model.get_input_embeddings()
    embed_dtype = embed_layer.weight.dtype
 
    virtual_block = soft_prompt.repeat(repeat, 1) if repeat > 1 else soft_prompt
    n_virtual_tokens = virtual_block.shape[0]
 
    bos_id = tokenizer.bos_token_id
    if bos_id is None:
        bos_id = tokenizer.eos_token_id  
 
    for data in eval_data:
        prompt_text = build_conflict_prompt(data)
        real_input_ids = tokenizer(
            prompt_text, return_tensors="pt", add_special_tokens=False
        ).input_ids.to(device)
        batch_size = real_input_ids.shape[0]
 
        bos_tensor = torch.full((batch_size, 1), bos_id, device=device, dtype=torch.long)
        bos_embeds = embed_layer(bos_tensor)
        real_embeds = embed_layer(real_input_ids)
        virtual_embeds = virtual_block.unsqueeze(0).expand(batch_size, -1, -1).to(embed_dtype)
 
        inputs_embeds = torch.cat([bos_embeds, virtual_embeds, real_embeds], dim=1)
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)
 
        with torch.no_grad():
            generated_ids = model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                cache_implementation="dynamic",
            )
        output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        counts[classify_generation(output, data["org_answer"], data["sub_answer"])] += 1
        print("output->",output)
        print(counts)
        del inputs_embeds, real_embeds, virtual_embeds, bos_embeds
        gc.collect()
        torch.cuda.empty_cache()
 
    return counts

In [ ]:
#Trains soft prompts across several (memory heads, context heads, prompt length) configurations and saves the checkpoints and results.
def run_training_sweep(
    model,
    configs,
    total_diff_mem: torch.Tensor,
    total_diff_context: torch.Tensor,
    train_data,
    val_data,
    role: str,  # "mem": train soft prompt to target mem-heads. "context": target context-heads.
    postname,
    checkpoint_dir: str = ".",
    save_json_path: str = None,
    num_of_epochs=1,
    compute_combined_loss=compute_combined_loss
):

    if role not in ("mem", "context"):
        raise ValueError('role must be "mem" or "context"')
 
    os.makedirs(checkpoint_dir, exist_ok=True)
    results = []
 
    for k_heads_mem, k_heads_context, prompt_len in configs:
        gc.collect()
        torch.cuda.empty_cache()
        print(
            "k_heads_mem=", k_heads_mem,
            "k_heads_context=", k_heads_context,
            "prompt_len=", prompt_len,
        )
 
        heads_mem = top_k_positive_heads(total_diff_mem, k_heads_mem)
        heads_context = top_k_positive_heads(total_diff_context, k_heads_context)
 
        if role == "mem":
            target_heads = [tuple(x) for x in heads_mem.tolist()]
            target_heads_context = [tuple(x) for x in heads_context.tolist()]
        else:
            target_heads = [tuple(x) for x in heads_context.tolist()]
            target_heads_context = [tuple(x) for x in heads_mem.tolist()]
 
        # One checkpoint file per (role, k_heads_mem, k_heads_context,
        # prompt_len) so turns don't overwrite each other.
        # `train_soft_prompt` uses this path to save its best-so-far
        # tensor during training and reload it at the end.
        checkpoint_name = (
            f"soft_prompt_role-{role}_kmem{k_heads_mem}_kctx{k_heads_context}_len{prompt_len}_{postname}.pt"
        )
        checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)
 
        soft_prompt, train_loss, val_loss = train_soft_prompt(
            model,
            train_data,
            val_data,
            target_heads,
            target_heads_context,
            n_virtual_tokens=prompt_len,
            checkpoint_path=checkpoint_path,
            num_of_epochs=num_of_epochs,
            compute_combined_loss=compute_combined_loss)
 

        torch.save(
            {
                "soft_prompt": soft_prompt.detach().cpu(),
                "role": role,
                "k_heads_mem": k_heads_mem,
                "k_heads_context": k_heads_context,
                "prompt_len": prompt_len,
            },
            checkpoint_path,
        )
 
        turn_results = {
            "k_heads_mem&k_heads_context&prompt_len": (k_heads_mem, k_heads_context, prompt_len),
            "checkpoint_path": checkpoint_path,
            "loss_train": train_loss,
            "loss_val": val_loss,
        }
        results.append(turn_results)
 
        if save_json_path is not None:
            with open(save_json_path, "w") as f:
                json.dump(results, f)
 
    return results
 

In [ ]:
#Evaluates random-token prompts of each length as the control.
def evaluate_random_prompt_baseline(model,prompt_lens,eval_data,save_json_path= None,generation_repeat = 1,seed= 42):

    results: List[dict] = []
 
    for prompt_len in prompt_lens:
        gc.collect()
        torch.cuda.empty_cache()
        print("prompt_len=", prompt_len)
 
        rand_ids = random_token_ids(model.cfg.d_vocab, prompt_len, seed=seed, device=model.cfg.device)
        random_counts = evaluate_conflict_completions(
            model, rand_ids, eval_data, repeat=generation_repeat
        )
 
        turn_results = {
            "prompt_len": prompt_len,
            "random_prompt_count_from_mem": random_counts["mem"],
            "random_prompt_count_from_contxt": random_counts["context"],
            "random_prompt_count_no_from_mem_contxt": random_counts["neither"],
            "random_org_mem_answer_list_len": len(eval_data),
        }
        results.append(turn_results)
 
        if save_json_path is not None:
            with open(save_json_path, "w") as f:
                json.dump(results, f)
 
    return results

In [ ]:
#Loads checkpoints, converts the soft prompts to tokens, and evaluates them.
def evaluate_soft_prompt_checkpoints(model,checkpoint_paths,eval_data,save_json_path= None,generation_repeat: int = 1):

    results: List[dict] = []
 
    for checkpoint_path in checkpoint_paths:
        gc.collect()
        torch.cuda.empty_cache()
 
        checkpoint = torch.load(checkpoint_path)
        soft_prompt = checkpoint["soft_prompt"].to(model.cfg.device).to(model.cfg.dtype)
        k_heads_mem = checkpoint["k_heads_mem"]
        k_heads_context = checkpoint["k_heads_context"]
        prompt_len = checkpoint["prompt_len"]
        print(
            "checkpoint=", checkpoint_path,
            "k_heads_mem=", k_heads_mem,
            "k_heads_context=", k_heads_context,
            "prompt_len=", prompt_len,
        )
 
        soft_ids = soft_prompt_to_ids(soft_prompt, model, k=5)
        soft_counts = evaluate_conflict_completions(
            model, soft_ids, eval_data, repeat=generation_repeat
        )
 
        turn_results = {
            "k_heads_mem&k_heads_context&prompt_len": (k_heads_mem, k_heads_context, prompt_len),
            "checkpoint_path": checkpoint_path,
            "soft_prompt_count_from_mem": soft_counts["mem"],
            "soft_prompt_count_from_contxt": soft_counts["context"],
            "soft_prompt_count_no_from_mem_contxt": soft_counts["neither"],
            "len_org_mem_answer_list_len": len(eval_data),
        }
        results.append(turn_results)
 
        if save_json_path is not None:
            with open(save_json_path, "w") as f:
                json.dump(results, f)
 
    return results

In [ ]:
#Evaluates checkpoints using the continuous embeddings on a HuggingFace model, to measure the cost of discretization.
def evaluate_soft_prompt_checkpoints_with_embeddings_hf(model,tokenizer,checkpoint_paths,eval_data,save_json_path= None,generation_repeat = 1):

    results: List[dict] = []
    device = model.device
    embed_dtype = model.get_input_embeddings().weight.dtype
 
    for checkpoint_path in checkpoint_paths:
        gc.collect()
        torch.cuda.empty_cache()
 
        checkpoint = torch.load(checkpoint_path)
        soft_prompt = checkpoint["soft_prompt"].to(device=device, dtype=embed_dtype)
        k_heads_mem = checkpoint["k_heads_mem"]
        k_heads_context = checkpoint["k_heads_context"]
        prompt_len = checkpoint["prompt_len"]
        print(
            "checkpoint=", checkpoint_path,
            "k_heads_mem=", k_heads_mem,
            "k_heads_context=", k_heads_context,
            "prompt_len=", prompt_len,
        )
 
        soft_counts = evaluate_conflict_completions_with_embeddings_hf(
            model, tokenizer, soft_prompt, eval_data, repeat=generation_repeat
        )
 
        turn_results = {
            "k_heads_mem&k_heads_context&prompt_len": (k_heads_mem, k_heads_context, prompt_len),
            "checkpoint_path": checkpoint_path,
            "soft_prompt_count_from_mem": soft_counts["mem"],
            "soft_prompt_count_from_contxt": soft_counts["context"],
            "soft_prompt_count_no_from_mem_contxt": soft_counts["neither"],
            "len_org_mem_answer_list_len": len(eval_data),
        }
        results.append(turn_results)
 
        if save_json_path is not None:
            with open(save_json_path, "w") as f:
                json.dump(results, f)
 
    return results
 

In [ ]:
"""training"""

In [ ]:
configs_mem_Steer_with_embedd = [(15, 15, 25)]#[(15, 15, 300), (15, 15, 500)]
configs_context_Steer_with_embedd = [(3, 50, 25)]#[(3, 50, 300), (3, 50, 500)]
train_HF_with_embedd, val_HF_with_embedd = prepare_squad_datasets( seed = 42,
    max_context_words = 100,
    n_train = 500,
    n_val = 10,)

ctx_training_results_Steer_with_embedd = run_training_sweep(
    model=model,  # noqa: F821 - provided by the surrounding notebook
    configs=configs_context_Steer_with_embedd,
    total_diff_mem=total_diff_MEM_A,  # noqa: F821
    total_diff_context=total_diff_context_A,  # noqa: F821
    train_data=train_HF_with_embedd,
    val_data=val_HF_with_embedd,
    role="context",
    postname="_with_embedd",
    checkpoint_dir="/kaggle/working/",
    save_json_path="/kaggle/working/training_results_context_prop_with_embedd.json",
    compute_combined_loss=compute_combined_loss_4
)

mem_training_results_Steer_with_embedd = run_training_sweep(
    model=model,  # noqa: F821 - provided by the surrounding notebook
    configs=configs_mem_Steer_with_embedd,
    total_diff_mem=total_diff_MEM_A,  # noqa: F821
    total_diff_context=total_diff_context_A,  # noqa: F821
    train_data=train_HF_with_embedd,
    val_data=val_HF_with_embedd,
    role="mem",
    postname="_with_embedd",
    checkpoint_dir="/kaggle/working/",
    save_json_path="/kaggle/working/training_results_mem_prop_with_embedd_token_id_form_q.json",
    num_of_epochs=1,
    compute_combined_loss=compute_combined_loss_4
)

In [ ]:
"""testing"""

In [ ]:
#Creates random soft-prompt checkpoints, used as the random-embedding control.
def create_random_embedding_checkpoints(hf_model,prompt_lens,checkpoint_dir = "/kaggle/working/checkpoints_random_embed",seed = 42,scale= 0.02):

    os.makedirs(checkpoint_dir, exist_ok=True)
    hidden_dim = hf_model.config.hidden_size
    g = torch.Generator().manual_seed(seed)

    checkpoint_paths: List[str] = []
    for prompt_len in prompt_lens:
        soft_prompt = torch.randn(prompt_len, hidden_dim, generator=g) * scale

        checkpoint_path = os.path.join(checkpoint_dir, f"random_embed_len{prompt_len}.pt")
        torch.save(
            {
                "soft_prompt": soft_prompt,
                "k_heads_mem": None,
                "k_heads_context": None,
                "prompt_len": prompt_len,
            },
            checkpoint_path,
        )
        checkpoint_paths.append(checkpoint_path)

    return checkpoint_paths


In [ ]:
shorter_100_list_of_memory_answers_data=list_of_memory_answers_data[:100]
shorter_100_list_of_context_answers_data=list_of_context_answers_data[:100]

#soft to hard prompt
checkpoint_paths_mem_with_embbed=[r"/kaggle/input/datasets/nanaenana/soft-prpmpt-and-train-curve-25-9-ds/soft_prompt_role-mem_kmem15_kctx15_len300__with_embedd (2).pt",r"/kaggle/input/datasets/nanaenana/soft-prpmpt-and-train-curve-25-9-ds/soft_prompt_role-mem_kmem15_kctx15_len500__with_embedd_fix_2.pt"]

soft_results_mem_hard_prompt = evaluate_soft_prompt_checkpoints(
    model=model,  # noqa: F821
    checkpoint_paths=checkpoint_paths_mem_with_embbed,
    eval_data=shorter_100_list_of_memory_answers_data,  
    save_json_path="/kaggle/working/soft_results_mem_hard_prompt.json",
)

checkpoint_paths_context_with_embbed=[r"/kaggle/working/soft_prompt_role-context_kmem3_kctx50_len300__with_embedd.pt",r"/kaggle/working/soft_prompt_role-context_kmem3_kctx50_len500__with_embedd.pt"]

soft_results_context_hard_prompt=evaluate_soft_prompt_checkpoints(
    model=model,
    checkpoint_paths=checkpoint_paths_context_with_embbed,
    eval_data=shorter_100_list_of_context_answers_data,
    save_json_path="/kaggle/working/soft_results_context_hard_prompt.json",
)

#random hard prompt
random_hrad_prompt_results_mem_prop_300_500 = evaluate_random_prompt_baseline(
    model=model,  # noqa: F821
    prompt_lens=[300,500],
    eval_data=shorter_100_list_of_memory_answers_data,  
    save_json_path="/kaggle/working/random_hrad_prompt_results_mem_prop_300_500.json",
)

random_hrad_prompt_results_context_prop_300_500 = evaluate_random_prompt_baseline(
    model=model,  # noqa: F821
    prompt_lens=[300,500],
    eval_data=shorter_100_list_of_context_answers_data,  
    save_json_path="/kaggle/working/random_hrad_prompt_results_context_prop_300_500.json",
)

In [ ]:
model.reset_hooks()
del model
gc.collect()
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

In [ ]:
shorter_list_of_memory_answers_data=list_of_memory_answers_data[:50]
shorter_list_of_context_answers_data=list_of_context_answers_data[:50]
#testing random soft prompt
random_embedding_checkpoints=create_random_embedding_checkpoints(
    hf_model=hf_model,  
    prompt_lens=[50, 300, 500],  
    checkpoint_dir="/kaggle/working/checkpoints_random_embed",
)

hf_soft_random_mem = evaluate_soft_prompt_checkpoints_with_embeddings_hf(
    model=hf_model,  
    tokenizer=tokenizer,  
    checkpoint_paths=random_embedding_checkpoints,
    eval_data=shorter_list_of_memory_answers_data,  
    save_json_path="/kaggle/working/hf_soft_results_soft_random_mem_50-300-500.json",
)

hf_soft_random_context = evaluate_soft_prompt_checkpoints_with_embeddings_hf(
    model=hf_model,  
    tokenizer=tokenizer,  
    checkpoint_paths=random_embedding_checkpoints,
    eval_data=shorter_list_of_context_answers_data,  
    save_json_path="/kaggle/working/hf_soft_results_soft_random_context_50-300-500.json",
)


In [ ]:
shorter_100_list_of_memory_answers_data=list_of_memory_answers_data[:100]
shorter_100_list_of_context_answers_data=list_of_context_answers_data[:100]

#testing learned soft prompt
checkpoint_paths_mem_with_embbed=[r"/kaggle/input/datasets/nanaenana/soft-prpmpt-and-train-curve-25-9-ds/soft_prompt_role-mem_kmem15_kctx15_len300__with_embedd (2).pt",r"/kaggle/input/datasets/nanaenana/soft-prpmpt-and-train-curve-25-9-ds/soft_prompt_role-mem_kmem15_kctx15_len500__with_embedd_fix_2.pt"]
hf_soft_mem_with_emmbed = evaluate_soft_prompt_checkpoints_with_embeddings_hf(
    model=hf_model,  
    tokenizer=tokenizer,  
    checkpoint_paths=checkpoint_paths_mem_with_embbed,
    eval_data=shorter_100_list_of_memory_answers_data,  # noqa: F821
    save_json_path="/kaggle/working/hf_soft_results_hf_soft_mem_with_emmbed_token_id_form_q..json",
)


checkpoint_paths_context_with_embbed=[r"/kaggle/working/soft_prompt_role-context_kmem3_kctx50_len300__with_embedd.pt",r"/kaggle/working/soft_prompt_role-context_kmem3_kctx50_len500__with_embedd.pt"]
hf_soft_context_with_emmbed = evaluate_soft_prompt_checkpoints_with_embeddings_hf(
    model=hf_model,  
    tokenizer=tokenizer,  
    checkpoint_paths=checkpoint_paths_context_with_embbed,
    eval_data=shorter_100_list_of_context_answers_data,  
    save_json_path="/kaggle/working/hf_soft_results_hf_soft_context_with_emmbed_token_id_form_q..json",
)

